# QuantumCircuit.jl Phase 3A Circuit Hamiltonian Dynamics

**Audience**
- Julia users who already know the static and sweep workflows and want a dedicated charge-basis walkthrough.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `CompositeSystem`, `spectrum`, and `simulate_sweep`.

**What you will learn**
- How to build supported uncoupled circuit-mode models with `CircuitHamiltonianSpec`.
- How circuit-mode charge cutoff differs from the effective-model `ncut` parameter.
- How to use circuit-native operators such as `charge`, `cosphi`, and `sinphi`.
- How to run closed-system circuit-mode dynamics with a local `:sinphi` drive.
- How to apply `FluxControl` to a supported uncoupled tunable device in circuit mode.


## Outline

1. Activate the local package environment.
2. Review the current circuit-mode support matrix and limits.
3. Build uncoupled `Transmon`, `TunableTransmon`, and `TunableCoupler` systems in circuit mode.
4. Check how the charge cutoff affects a small spectrum calculation.
5. Inspect circuit-native operators and run a local `:sinphi` drive example.
6. Run a tunable circuit-mode example with a static flux sweep and one `FluxControl` dynamics run.
7. Review the current limits, then try a short exercise.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using QuantumToolbox: expect
using UnicodePlots: lineplot, lineplot!


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


## Step 1 - Circuit-mode support matrix and current limits

This notebook stays inside the currently supported circuit-Hamiltonian surface.

| Workflow | Supported in Phase 3A? | Notes |
| --- | --- | --- |
| Uncoupled `CircuitHamiltonianSpec(charge_cutoff = ...)` | Yes | Charge-basis dynamics for transmon-like subsystems. |
| Circuit-native observables `charge`, `cosphi`, `sinphi` | Yes | Valid for transmon-like subsystems in circuit mode. |
| `basis_state(...)` and `evolve(...)` in circuit mode | Yes | Works for supported uncoupled systems. |
| `FluxControl(...)` in circuit mode | Yes | Current support is limited to uncoupled tunable devices. |
| Circuit-mode capacitive couplings | No | Use effective mode for coupled systems today. |
| Coupled circuit-mode dynamics examples | No | Intentionally deferred until a dedicated circuit-coupling model exists. |


## Step 2 - Build uncoupled circuit-mode models

Circuit mode uses a separate charge cutoff on `CircuitHamiltonianSpec`. The subsystem `ncut` values still belong to the effective-model path, so the charge-basis Hilbert dimension is controlled independently here.


In [3]:
circuit_spec = CircuitHamiltonianSpec(charge_cutoff = 3)

q_static = Transmon(:q_static; EJ = 20.0, EC = 0.25, ng = 0.10, ncut = 6)
tq_flux = TunableTransmon(:tq_flux; EJmax = 20.0, EC = 0.25, flux = 0.12, asymmetry = 0.20, ng = 0.05, ncut = 6)
c_flux = TunableCoupler(:c_flux; EJmax = 15.0, EC = 0.30, flux = 0.08, asymmetry = 0.10, ng = 0.0, ncut = 5)

q_sys = CompositeSystem(q_static)
tq_sys = CompositeSystem(tq_flux)
c_sys = CompositeSystem(c_flux)

q_model = build_model(q_sys; hamiltonian_spec = circuit_spec)
tq_model = build_model(tq_sys; hamiltonian_spec = circuit_spec)
c_model = build_model(c_sys; hamiltonian_spec = circuit_spec)

(; charge_cutoff = circuit_spec.charge_cutoff,
   static_dim = q_model.dimensions[:q_static],
   tunable_transmon_dim = tq_model.dimensions[:tq_flux],
   tunable_coupler_dim = c_model.dimensions[:c_flux])


(charge_cutoff = 3, static_dim = 7, tunable_transmon_dim = 7, tunable_coupler_dim = 7)

## Step 3 - Charge-cutoff choice and a small convergence check

A larger circuit-mode charge cutoff increases Hilbert dimension and cost, so it is useful to inspect how sensitive a low-lying transition is before running longer dynamics.


In [4]:
function circuit_transition_01(system::CompositeSystem, cutoff::Int)
    spec = spectrum(system; levels = 4, hamiltonian_spec = CircuitHamiltonianSpec(charge_cutoff = cutoff))
    return transition_frequencies(spec)[1]
end

cutoff_scan = [
    (; charge_cutoff = cutoff, transition_01 = circuit_transition_01(tq_sys, cutoff))
    for cutoff in 2:5
]

(; cutoff_scan,
   final_change = cutoff_scan[end].transition_01 - cutoff_scan[end - 1].transition_01)


(cutoff_scan = [(charge_cutoff = 2, transition_01 = 8.142541119926095), (charge_cutoff = 3, transition_01 = 6.334192542456369), (charge_cutoff = 4, transition_01 = 5.90706701635909), (charge_cutoff = 5, transition_01 = 5.850403016206453)], final_change = -0.05666400015263662)

## Step 4 - Circuit-native operators and observables

Circuit mode exposes charge-basis operators directly. They can be inspected as operators or requested through `ObservableSpec` during evolution.


In [5]:
psi_basis = basis_state(q_sys; hamiltonian_spec = circuit_spec, q_static = 2)

charge_q = charge_operator(q_model, :q_static)
cosphi_q = cosphi_operator(q_model, :q_static)
sinphi_q = sinphi_operator(q_model, :q_static)

(; operator_sizes = (
       charge = size(charge_q.data),
       cosphi = size(cosphi_q.data),
       sinphi = size(sinphi_q.data),
   ),
   basis_length = length(psi_basis.data),
   basis_expectations = (
       charge = expect(charge_q, psi_basis),
       cosphi = expect(cosphi_q, psi_basis),
       sinphi = expect(sinphi_q, psi_basis),
   ))


(operator_sizes = (charge = (7, 7), cosphi = (7, 7), sinphi = (7, 7)), basis_length = 7, basis_expectations = (charge = -1.0 + 0.0im, cosphi = 0.0 + 0.0im, sinphi = 0.0 + 0.0im))

## Step 5 - Circuit-mode basis states and closed-system dynamics with a local `:sinphi` drive

Here we start from a named circuit-mode basis state and drive the fixed transmon through the `:sinphi` operator. The observable requests stay in the circuit-native symbol set.


In [6]:
tlist = collect(range(0.0, 8.0; length = 161))

sinphi_drive = SubsystemDrive(
    :q_static_phi_drive,
    :q_static,
    :sinphi,
    (p, t) -> p.Omega * cos(p.omega_d * t),
)

circuit_dynamics = evolve(
    q_sys,
    psi_basis,
    tlist;
    hamiltonian_spec = circuit_spec,
    drives = [sinphi_drive],
    observables = [
        ObservableSpec(:charge_q, :q_static, :charge),
        ObservableSpec(:cosphi_q, :q_static, :cosphi),
    ],
    params = (; Omega = 0.20, omega_d = 1.0),
)

charge_trace = observable_trace(circuit_dynamics, :charge_q)
cosphi_trace = observable_trace(circuit_dynamics, :cosphi_q)
level1_population = population_trace(circuit_dynamics, :q_static, 1)

(; first_times = charge_trace.times[1:8],
   first_charge_values = charge_trace.values[1:8],
   max_level1_population = maximum(level1_population.values))


(first_times = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35], first_charge_values = ComplexF64[-1.0 + 0.0im, -0.9993005364802524 + 4.235164736271502e-22im, -0.9803787901085235 + 2.7755575615628914e-17im, -0.8675659681831538 + 0.0im, -0.5734920049323876 + 0.0im, -0.1470653742422442 + 4.163336342344337e-17im, 0.23193264183742734 - 5.551115123125783e-17im, 0.44941590190505776 + 0.0im], max_level1_population = 0.42453985613827033)

In [7]:
drive_plot = lineplot(
    charge_trace.times,
    real.(charge_trace.values);
    name = "charge",
    title = "Circuit-mode local sinphi drive",
    xlabel = "time",
    ylabel = "expectation",
)
lineplot!(drive_plot, cosphi_trace.times, real.(cosphi_trace.values); name = "cosphi")
drive_plot


                  ⠀⠀⠀⠀⠀⠀Circuit-mode local sinphi drive⠀⠀⠀⠀⠀       
                  ┌────────────────────────────────────────┐       
                2 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ charge
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ cosphi
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│       
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⡆⠀⠀⠀│       
                  │⠀⠀⠀⠀⠀⢀⡴⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡇⡇⠀⠀⢀│       
                  │⠀⢠⢢⠀⠀⢸⠀⢧⠀⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡶⡄⠀⠀⣠⡀⠀⠀⢰⢳⠀⠀⢸⠀⢹⠀⠀⢸│       
                  │⢀⣼⢸⡀⢀⣼⠀⢸⠀⣀⠞⢣⣀⠀⠀⣠⡀⠀⢀⣀⡀⣼⡁⣇⡀⣠⣇⣇⡀⢠⣜⠀⣇⢀⣸⠀⣸⡀⢀⡏│       
   expectation    │⠯⡯⠭⡿⠮⡯⠷⢽⡭⢼⠷⢽⠬⠭⠭⡥⠬⡭⠯⢵⢵⠯⠬⢽⠵⢧⠯⢿⠭⠥⡯⠾⡧⠭⡯⠾⢼⠵⢥⠿│       
                  │⠀⡇⠀⢳⠀⡇⠀⠀⡇⡇⠀⠸⡄⢀⠞⢸⢰⠃⠀⠘⠇⠀⠀⢸⡀⢸⠀⢸⠀⠀⡇⠀⢹⠀⡇⠀⠘⡄⢸⠀│       
                  │⢀⠇⠀⠸⡄⡇⠀⠀⢳⠇⠀⠀⢇⡏⠀⠈⠋⠀⠀⠀⠀⠀⠀⠀⠉⠉⠀⠀⡇⢀⠇⠀⢸⠀⡇⠀⠀⣇⡞⠀│       
                  │⢸⠀⠀⠀⠻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢣⢸⠀⠀⠘⣼⠀⠀⠀⠈⠁⠀│       
                  │⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⡼⠀⠀⠀⠉⠀⠀⠀⠀⠀⠀│       
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Step 6 - Tunable circuit-mode example: static flux, sweep, and `FluxControl`

Now switch to an uncoupled tunable transmon. We first inspect its current operating point and a short static flux sweep, then run one supported circuit-mode `FluxControl` example.


In [8]:
tq_spec = spectrum(tq_sys; levels = 4, hamiltonian_spec = circuit_spec)
tq_sweep = simulate_sweep(
    tq_sys,
    SweepSpec(:tq_flux, :flux, [0.0, 0.12, 0.24, 0.36]; levels = 4);
    hamiltonian_spec = circuit_spec,
)
tq_summary = sweep_summary(tq_sweep)

(; low_lying_energies = tq_spec.energies,
   flux_values = tq_summary.values,
   transition_01 = tq_summary.transition_01)


(low_lying_energies = [-15.584335025752974, -9.250142483296605, -2.385045291903949, 4.9181598062922], flux_values = [0.0, 0.12, 0.24, 0.36], transition_01 = [6.638032532995258, 6.334192542456369, 5.458498878795362, 4.103462770141176])

In [9]:
flux_tlist = collect(range(0.0, 16.0; length = 161))
flux_pulse = FluxControl(
    :tq_flux_pulse,
    :tq_flux,
    (p, t) -> p.delta * sin(p.omega_f * t);
    derivative = (p, t) -> p.delta * p.omega_f * cos(p.omega_f * t),
)

flux_result = evolve(
    tq_sys,
    basis_state(tq_sys; hamiltonian_spec = circuit_spec, tq_flux = 0),
    flux_tlist;
    hamiltonian_spec = circuit_spec,
    flux_controls = [flux_pulse],
    observables = [ObservableSpec(:charge_flux, :tq_flux, :charge)],
    params = (; delta = 0.08, omega_f = 5.6),
)

flux_charge = observable_trace(flux_result, :charge_flux)
flux_population = population_trace(flux_result, :tq_flux, 1)

(; max_excited_population = maximum(flux_population.values),
   first_charge_values = flux_charge.values[1:8])


(max_excited_population = 0.7016331284730939, first_charge_values = ComplexF64[-3.0 + 0.0im, -2.290599760802901 + 1.1102230246251565e-16im, -1.0458490735814228 + 0.0im, 0.08175138361930637 + 0.0im, 1.0699600728109593 - 5.551115123125783e-17im, 1.58928206352395 + 0.0im, 1.1115964973306403 - 5.637851296924623e-17im, -0.06199668105524192 - 1.3877787807814457e-17im])

In [10]:
flux_plot = lineplot(
    flux_population.times,
    flux_population.values;
    name = "P(level 1)",
    title = "Circuit-mode flux-control example",
    xlabel = "time",
    ylabel = "population",
)
flux_plot


                  ⠀⠀⠀⠀⠀Circuit-mode flux-control example⠀⠀⠀⠀           
                  ┌────────────────────────────────────────┐           
              0.8 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ P(level 1)
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⡄⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
   population     │⡇⠀⡀⠀⠀⡆⠀⡀⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⡇⢸⡇⠀⠀⡇⠀⡇⠀⠀⠀⠀⠀⠀⢸⡄⠀⠀⠀⠀⠀⠀⢀⡀⠀⠀⠀⢸⡇⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⡇⢸⡇⠀⠀⣷⠀⡇⠀⡆⠀⠀⠀⢰⣾⣧⡆⠀⠀⠀⠀⠀⢸⡇⠀⠀⠀⢸⡇⣸⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⡇⣿⡇⢠⠀⣿⠀⡇⢠⡇⢰⠀⠀⣸⣿⣿⡇⡄⢀⡀⣧⡀⢸⣧⡄⠀⠀⢸⡇⣿⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀│           
                  │⣇⣿⡇⣸⠀⣿⠀⡇⢸⣷⣼⠀⢰⣿⣿⣿⣧⡇⢸⣇⣿⣧⣼⣿⣷⢠⡇⢸⡇⣿⣤⣿⠀⠀⠀⠀⠀⠀⠀⠀│    

## Step 7 - Current limits, pitfalls, and one optional extension

**Supported here**
- Uncoupled `CircuitHamiltonianSpec(charge_cutoff = ...)` workflows.
- Circuit-native operators and observables.
- Named basis states, closed-system dynamics, and local circuit-mode drives.
- `FluxControl` on an uncoupled tunable device.

**Still out of scope**
- Circuit-mode capacitive couplings.
- Coupled circuit-mode dynamics examples.
- Mixing circuit-mode coupled systems with `FluxControl`.

**Common pitfall**
- If you see an operator-support error, check that the run actually uses `hamiltonian_spec = circuit_spec`. The circuit-native symbols `:charge`, `:cosphi`, and `:sinphi` are not available under the default effective Hamiltonian spec.

**Optional extension**
- Repeat the tunable-device example with `EffectiveHamiltonianSpec(NonadiabaticDuffingEffectiveMethod())` and compare the qualitative response against the supported circuit-mode run above.


## Exercise

Try one short follow-up before leaving the notebook:

1. Raise the circuit charge cutoff from `3` to `4` and compare the last two `transition_01` values again.
2. Replace the `:charge` observable in the flux-control example with `:cosphi`.
3. Start the tunable-device dynamics from `basis_state(...; tq_flux = 1)` instead of the ground state and compare the population trace.


In [12]:
exercise_spec = CircuitHamiltonianSpec(charge_cutoff = 4)
exercise_transition = circuit_transition_01(tq_sys, exercise_spec.charge_cutoff)

(; suggested_charge_cutoff = exercise_spec.charge_cutoff,
   transition_01 = exercise_transition,
   next_steps = [
       "swap :charge for :cosphi in the flux-control example",
       "rerun the tunable-device dynamics from tq_flux = 1",
   ])


(suggested_charge_cutoff = 4, transition_01 = 5.90706701635909, next_steps = ["swap :charge for :cosphi in the flux-control example", "rerun the tunable-device dynamics from tq_flux = 1"])